# Manuscript reproduction walkthrough

Run the repository analysis groups, inspect selected numerical results, and verify exported CSVs against fixed reference tables. Figures are rendered with the repository manuscript style and saved as PDFs. Coverage follows the reproduction driver. Several manuscript panels are still assembled separately.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "reproduce_manuscript.py").is_file()
     and (path / "styles" / "manuscript.mplstyle").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open the notebook from the repository root or its notebooks folder.")

output = root / "outputs" / "manuscript"
reference = root / "tests" / "reference_results"
groups = [
    "pareto", "model-validation", "interpretation", "experimental-validation",
    "extended-operation", "characterization", "transport",
]
reference_groups = {"pareto", "model-validation", "interpretation",
                    "extended-operation", "transport"}
use_tex = False  # Enable when the manuscript LaTeX dependencies are installed.

required = [root / "scripts" / "verify_results.py"]
if "transport" in groups:
    required.append(root / "data" / "literature" / "transport_reference_values.csv")
missing = [str(path.relative_to(root)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Add required repository files: " + ", ".join(missing))

## Generate manuscript outputs

The model validation and SHAP groups take longer than the experimental summaries. Edit `groups` above to run a subset. The driver writes a separate subdirectory for each selected group.

In [ ]:
import os

def run_script(name, *arguments):
    completed = subprocess.run(
        [sys.executable, str(root / "scripts" / name), *map(str, arguments)],
        cwd=root, capture_output=True, text=True, encoding="utf-8", errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print(completed.stderr.rstrip(), file=sys.stderr)
    completed.check_returncode()

style_flags = [] if use_tex else ["--no-tex"]
run_script("reproduce_manuscript.py", "--output-dir", output, "--only", *groups, *style_flags)

## Inspect reported results

The tables below show the held-out model errors, the five-hour ammonia balance, and the nitrogen-transport comparison when their groups are selected.

In [ ]:
for group, table in [
    ("model-validation", "cv_metrics.csv"),
    ("extended-operation", "summary.csv"),
    ("transport", "ratio_summary.csv"),
]:
    if group in groups:
        print(table)
        display(pd.read_csv(output / group / table, float_precision="round_trip"))

for group in groups:
    for figure in sorted((output / group).glob("*.pdf")):
        print(figure.relative_to(root))

## Verify numerical outputs

The verifier compares parsed CSV values exactly against the fixed reference tables.

In [ ]:
selected = [group for group in groups if group in reference_groups]
if selected:
    run_script("verify_results.py", "--output-dir", output,
               "--reference-dir", reference, "--only", *selected)
else:
    print("The selected groups have no numerical reference tables in this verifier.")